# Event pairing benchmark

**Question:** when two sources describe the same disaster, which method notices?

`monty:corr_id` cannot join sources (two sources give one disaster different ids), so a matcher has to.
This notebook scores three candidates on a small hand-labelled gold set of real Montandon events:

| Scorer | What it is |
|---|---|
| **Shipped heuristic** | the time-and-distance score the app uses today (`factory/events/disasters.py`), with its shared-ID shortcuts removed so it is judged on evidence alone |
| **Learned** | logistic regression on the same evidence plus hazard relation and place-name overlap, scored out-of-sample (each window by a model trained on the others) |
| **Decision model** | a System One model (e.g. Jev) via OpenRouter's Decisions API, asked from the records' words, dates and distances whether they are the same disaster. Needs `OPENROUTER_API_KEY` |

A fourth row, **heuristic → model**, is the proposed architecture: the heuristic settles clear pairs and the
model is asked only about the band between `band_low` and `band_high`. Widen the band if the heuristic is
confidently wrong outside it (the notebook prints how many pairs it settles wrongly).

**How to read it.** Labels come from `gold_disasters.yml`: one reviewer's grouping, made by reading the records,
never from a shared ID. The gold set is small (about 100 records), so differences of a few points are noise;
look for large gaps and for *which* pairs each scorer gets wrong. Thresholds marked *in-sample* are tuned on
the data they are scored on and are optimistic. Pairs are not independent (a record sits in many pairs), so
read the numbers as indicative.

Extend the gold set by adding item ids to the YAML; the notebook re-scores on every run. The run's catalog
comes from the Montandon event you select; the event itself is not used.

In [ ]:
# --- Reserved parameters (injected by the Notebook Factory; defaults keep the notebook runnable standalone) ---
run_id = ""
inputs_dir = ""
output_dir = "outputs"
area = {}                      # {id, code, name, level, country, country_name, bbox}
area_geojson = ""              # FeatureCollection with the target area
subdivisions_geojson = ""      # FeatureCollection with the aggregation subdivisions
stac_item = {}                 # the selected STAC item (dict), only its catalog is used
stac_item_json = ""
stac_collection = ""
stac_api_url = ""              # root of the item's catalog (Montandon)
stac_catalog_profile = ""      # "generic" | "montandon"
filters = {}                   # the run's filter tags
hazard = ""

# --- User parameters ---
block_days = 45                # candidates are records at most this many days apart
decision_models = "typesafe/jev-1.14-20261014"   # comma-separated OpenRouter model ids, empty to skip
max_model_calls = 3000         # cap on Decisions API calls (one per candidate pair per model)
l2 = 1.0                       # ridge strength of the learned scorer
band_low = 0.4                 # hybrid: heuristic scores below this are settled as different
band_high = 0.7                # hybrid: heuristic scores at or above this are settled as same


## 1. Gold set: the disasters and their records

In [ ]:
import json
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

sys.path.insert(0, str(Path.cwd()))  # the helper module ships next to the notebook
import pairing as P

warnings.filterwarnings("ignore")
Path(output_dir).mkdir(parents=True, exist_ok=True)

gold = yaml.safe_load(Path("gold_disasters.yml").read_text())
label, windows, kinds = P.gold_labels(gold)

api_url = stac_api_url or os.environ.get("STAC_API_URL", "")
if not api_url:
    raise RuntimeError("No catalog: select a Montandon event when starting the run (it supplies the catalog URL).")
items = P.fetch_items(api_url, os.environ.get("STAC_API_KEY", ""), list(label))
missing = sorted(set(label) - set(items))
label = {i: d for i, d in label.items() if i in items}
records = {i: P.record_from_item(x) for i, x in items.items()}

coverage = (
    pd.DataFrame({"disaster": [label[i] for i in records], "window": [windows[label[i]] for i in records]})
    .groupby(["window", "disaster"]).size().rename("records").reset_index()
)
print(f"{len(items)} of {len(items) + len(missing)} gold records found in the catalog; "
      f"{len(gold['ambiguous'])} more are listed as ambiguous and never scored.")
if missing:
    print("MISSING from the catalog (re-ingested or removed? update gold_disasters.yml):", ", ".join(missing))
display(coverage)


## 2. Candidate pairs

Stage 1 of any matcher is cheap and deterministic: only records that share a country (an unknown country never
blocks) and are at most `block_days` apart are candidates. Pairs inside one source family are skipped, and IDMC's
two collections count as one family. Positives are pairs from the same gold disaster; negatives are pairs from
different disasters of the same window (same country and period), which is what makes them hard.

In [ ]:
pairs = pd.DataFrame(P.build_pairs(records, gold, block_days))
total_pos = int(pairs.y.sum())
lost = pairs[(pairs.y == 1) & (pairs.block != "ok")]
blocked = pairs[pairs.block == "ok"].reset_index(drop=True)
y = blocked.y.to_numpy()

display(pd.DataFrame({
    "pairs": [len(pairs), len(blocked)],
    "positive": [total_pos, int(y.sum())],
    "negative": [int((pairs.y == 0).sum()), int((y == 0).sum())],
}, index=["all pairs in a window", "candidates (pass blocking)"]))
print(f"Blocking recall: {1 - len(lost) / max(total_pos, 1):.1%} "
      f"({len(lost)} positives lost; by reason: {lost.block.value_counts().to_dict() or 'none'})")
display(blocked.groupby("pair_type").y.agg(pairs="size", positives="sum"))
display(blocked.groupby("hazard_rel").y.agg(pairs="size", positives="sum")
        .rename_axis("hazard codes of the two records"))


## 3. Scorers

The learned scorer's weights (standardised features, fit on all candidates) show which evidence carries the
decision; positive means "more likely the same disaster".

In [ ]:
SHIPPED, LEARNED, HYBRID = "Shipped heuristic", "Learned", "Heuristic → model"
scores = {SHIPPED: blocked.heuristic.astype(float).to_numpy()}
default_threshold = {SHIPPED: P.STRONG, LEARNED: 0.5}

X = blocked[[f"f_{k}" for k in P.FEATURES]].to_numpy(float)
scores[LEARNED] = P.cross_val_logreg(X, y, blocked.window.to_numpy(), l2)

weights = P.fit_logreg(X, y, l2).weights[1:]
display(pd.Series(weights, index=P.FEATURES, name="weight").sort_values(key=abs, ascending=False).round(2).to_frame())


In [ ]:
import time

api_key = os.environ.get("OPENROUTER_API_KEY", "")
models = [m.strip() for m in decision_models.split(",") if m.strip()]
model_names, model_of, model_log = [], {}, []

if not models:
    print("No decision model requested: skipping.")
elif not api_key:
    print("OPENROUTER_API_KEY is not set: skipping the decision model.")
elif len(blocked) * len(models) > max_model_calls:
    print(f"{len(blocked) * len(models)} calls needed, above max_model_calls={max_model_calls}: skipping.")
else:
    pair_records = [(records[a], records[b]) for a, b in zip(blocked.a, blocked.b)]
    for model in models:
        started = time.time()
        results = P.score_with_model(pair_records, model, api_key)
        name = f"Model {model.split('/')[-1]}"
        model_names.append(name)
        model_of[name] = model
        blocked[f"p_same|{model}"] = [r["p_same"] for r in results]
        blocked[f"relation|{model}"] = [r.get("relation") for r in results]
        scores[name] = blocked[f"p_same|{model}"].astype(float).to_numpy()
        default_threshold[name] = 0.5
        errors = [r["error"] for r in results if r.get("error")]
        served = next((r["model"] for r in results if r.get("model")), None)
        model_log.append({"requested": model, "served": served, "pairs": len(results), "errors": len(errors),
                          "cost": round(sum(r.get("cost") or 0 for r in results), 5),
                          "seconds": round(time.time() - started)})
        if errors:
            print(f"{model}: {len(errors)} calls failed, e.g. {errors[0]}")
    display(pd.DataFrame(model_log))


In [ ]:
# The proposed architecture: the heuristic settles clear pairs, the model is asked about the band between.
h = scores[SHIPPED]
for name in model_names:
    p = scores[name]
    hybrid = np.where(h >= band_high, 1.0, np.where(h < band_low, 0.0, np.where(np.isnan(p), h, p)))
    scores[f"{HYBRID} {name.removeprefix('Model ')}"] = hybrid
    default_threshold[f"{HYBRID} {name.removeprefix('Model ')}"] = 0.5

band = (h >= band_low) & (h < band_high)
settled_wrong = int((((h >= band_high) & (y == 0)) | ((h < band_low) & (y == 1))).sum())
print(f"Escalated to the model: heuristic between {band_low} and {band_high}, {int(band.sum())} of {len(h)} candidates "
      f"({int(y[band].sum())} positive). Settled by the heuristic but wrong: {settled_wrong}, which no model can fix.")


## 4. Results

*Average precision* ignores thresholds: 1.0 means every same-disaster pair outranks every different-disaster
pair. *Precision / recall / F1* are at each scorer's default threshold (heuristic 0.7, as in the app; the others
0.5). *End-to-end recall* counts positives lost to blocking as misses.

In [ ]:
def evaluate(name):
    s = scores[name]
    ok = ~np.isnan(s)
    thr = default_threshold[name]
    p, r, f = P.prf(y[ok], s[ok] >= thr)
    best_thr, best = P.best_f1(y[ok], s[ok])
    tp = int(((s >= thr) & ok & (y == 1)).sum())
    return {"scorer": name, "pairs scored": int(ok.sum()), "average precision": P.average_precision(y[ok], s[ok]),
            "threshold": thr, "precision": p, "recall": r, "F1": f,
            "end-to-end recall": tp / max(total_pos, 1),
            "false merges": int(((s >= thr) & ok & (y == 0)).sum()),
            "best F1 (in-sample)": best, "at threshold": best_thr}

metrics = pd.DataFrame([evaluate(n) for n in scores]).set_index("scorer")
display(metrics.round(3))


In [ ]:
# Palette: validated categorical slots 1-4 (blue, orange, aqua, yellow) in a fixed order; aqua and yellow are
# low-contrast on white, so every line is also labelled directly and the table above gives the same numbers.
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
def short(name):
    return "Decision model" if name.startswith("Model ") else HYBRID if name.startswith(HYBRID) else name


shown = [SHIPPED, LEARNED] + model_names[:1] + [n for n in scores if n.startswith(HYBRID)][:1]
fig, ax = plt.subplots(figsize=(7.5, 4.6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ends = []
for name, color in zip(shown, SLOTS, strict=False):
    ok = ~np.isnan(scores[name])
    thr, precision, recall = P.pr_curve(y[ok], scores[name][ok])
    ax.plot(recall, precision, color=color, lw=2, label=short(name), zorder=2)
    i = int(np.argmin(np.abs(thr - default_threshold[name])))
    ax.plot(recall[i], precision[i], "o", ms=8, color=color, mec=SURFACE, mew=2, zorder=3)
    ends.append([name, color, float(precision[-1])])
ends.sort(key=lambda e: e[2])
for k in range(1, len(ends)):  # nudge end labels apart
    ends[k][2] = max(ends[k][2], ends[k - 1][2] + 0.06)
for name, _color, yy in ends:
    ax.annotate(short(name), (1.0, yy), xytext=(6, 0), textcoords="offset points", va="center",
                fontsize=9, color=INK2, annotation_clip=False)
ax.axhline(y.mean(), color=INK2, lw=1, ls=(0, (4, 3)), zorder=1)
ax.annotate("chance", (0.0, y.mean()), xytext=(4, 4), textcoords="offset points", fontsize=8, color=INK2)
ax.set(xlim=(0, 1), ylim=(0, 1.03), xlabel="Recall", ylabel="Precision")
ax.set_title("Precision-recall on candidate pairs (dots: default threshold)", loc="left", fontsize=11, color=INK)
ax.grid(color=GRID, lw=0.8, zorder=0)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color(GRID)
ax.tick_params(colors=INK2, labelsize=9)
ax.xaxis.label.set_color(INK2)
ax.yaxis.label.set_color(INK2)
ax.legend(loc="lower left", frameon=False, fontsize=9, labelcolor=INK2)
fig.subplots_adjust(right=0.78)
plt.show()


## 5. Where each scorer fails

Rows are slices of the candidate pairs; each scorer shows `recall / false merges` at its default threshold.
*Hazard codes differ* holds cascades (a cyclone and its floods); *coarse* sources give only a country and a day.

In [ ]:
slices = {
    "all candidates": np.ones(len(blocked), bool),
    "coarse - coarse": (blocked.pair_type == "coarse-coarse").to_numpy(),
    "coarse - precise": (blocked.pair_type == "coarse-precise").to_numpy(),
    "precise - precise": (blocked.pair_type == "precise-precise").to_numpy(),
    "hazard codes differ (cascade)": blocked.hazard_rel.isin(["cascade", "different"]).to_numpy(),
}
rows = {}
for slice_name, m in slices.items():
    row = {"positives": int((y[m] == 1).sum()), "negatives": int((y[m] == 0).sum())}
    for name, s in scores.items():
        pred = (s >= default_threshold[name]) & ~np.isnan(s)
        row[name] = f"{pred[m & (y == 1)].sum() / max((y[m] == 1).sum(), 1):.0%} / {int(pred[m & (y == 0)].sum())}"
    rows[slice_name] = row
display(pd.DataFrame(rows).T)


In [ ]:
titles = {i: r.title[:48] for i, r in records.items()}
view = blocked[["a", "b", "y"]].assign(title_a=blocked.a.map(titles), title_b=blocked.b.map(titles)).join(pd.DataFrame(scores))
decided = {n: (scores[n] >= default_threshold[n]) for n in scores}
wrong = pd.Series(decided[SHIPPED] != (y == 1))
cols = ["title_a", "title_b", "y"] + [n for n in scores if not n.startswith(HYBRID)]

print(f"Heuristic false merges ({int((wrong & (y == 0)).sum())} in all), most confident first:")
display(view[wrong & (y == 0)].sort_values(SHIPPED, ascending=False)[cols].head(12).round(2))
print(f"Heuristic misses ({int((wrong & (y == 1)).sum())} in all), lowest score first:")
display(view[wrong & (y == 1)].sort_values(SHIPPED)[cols].head(12).round(2))


In [ ]:
# Does the model fix what the heuristic gets wrong, and break what it gets right?
for name in model_names:
    ok = ~np.isnan(scores[name])
    heuristic_right = (decided[SHIPPED] == (y == 1))[ok]
    model_right = (decided[name] == (y == 1))[ok]
    print(name)
    display(pd.crosstab(pd.Series(heuristic_right, name="heuristic"), pd.Series(model_right, name="model"))
            .rename(index={True: "right", False: "wrong"}, columns={True: "right", False: "wrong"}))

    # What a deployment would do with the model: link / drop / send to a person.
    verdict = [P.gate(None if np.isnan(p) else p) for p in scores[name]]
    gates = pd.crosstab(pd.Series(verdict, name=f"gate ({P.SAME_NO} / {P.SAME_YES})"),
                        pd.Series(np.where(y == 1, "same disaster", "different"), name="truth"))
    display(gates)

    relation_col = f"relation|{model_of[name]}"
    cascades = blocked[blocked.kind.eq("cascade") & (blocked.y == 1)]
    if relation_col in blocked and len(cascades):
        print("Relation the model gives to positive pairs of cascade disasters:")
        display(cascades[relation_col].value_counts(dropna=False).to_frame("pairs"))


## 6. Outputs

In [ ]:
out = Path(output_dir)
scored = pd.DataFrame(scores).join(blocked[["a", "b"] + [c for c in blocked if c.startswith("relation|")]])
(pairs.merge(scored, on=["a", "b"], how="left")
      .assign(title_a=lambda d: d.a.map(titles), title_b=lambda d: d.b.map(titles))
      .to_csv(out / "pair_scores.csv", index=False))
metrics.round(4).to_csv(out / "metrics.csv")
(out / "run_summary.json").write_text(json.dumps({
    "records": len(records), "missing": missing, "pairs": len(pairs), "candidates": len(blocked), "positives": total_pos,
    "blocking_recall": 1 - len(lost) / max(total_pos, 1), "block_days": block_days, "models": model_log,
}, indent=2))
print("wrote", ", ".join(sorted(p.name for p in out.iterdir())))


## What to do with this

* A scorer that wins on *coarse - coarse* and *hazard codes differ* rows is the one worth wiring into the ETL;
  one that only wins on pairs a time-and-distance rule already gets right is not.
* Read the false merges and misses tables first. Each row is either a scorer weakness or a wrong label: fix the
  label in `gold_disasters.yml`, or fix the scorer.
* Before trusting the decision model's thresholds (`SAME_YES`, `SAME_NO` in `pairing.py`), set them from the
  `gate` table above on a larger gold set, and re-check whenever the model version changes.
* The `pair_scores.csv` output has every pair with every score, for review in a spreadsheet.